In [ ]:
GWAS_REPO = "/home/rodrigo/01_repos/GWAS_pipeline/"
CARDIAC_COMA_REPO = "/home/rodrigo/01_repos/CardiacCOMA/"
CARDIAC_GWAS_REPO = "/home/rodrigo/01_repos/CardiacGWAS/"

In [ ]:
import mlflow
import os, sys

#import torch
#import torch.nn.functional as F

import os; os.chdir(CARDIAC_COMA_REPO)
from config.load_config import load_yaml_config, to_dict

import ipywidgets as widgets
from ipywidgets import interact
from IPython.display import Image
from mlflow.tracking import MlflowClient

import pickle as pkl
import pytorch_lightning as pl

from argparse import Namespace
import matplotlib.pyplot as plt

# import surgeon_pytorch
# from surgeon_pytorch import Inspect, get_layers

import numpy as np
import pandas as pd
from IPython import embed
sys.path.insert(0, '..')

import model.Model3D
# from utils.helpers import get_coma_args, get_lightning_module, get_datamodule
from copy import deepcopy
from pprint import pprint

from copy import deepcopy
from typing import List
from tqdm import tqdm
from IPython import embed

In [ ]:
def fetch_loci_mapping(link=None):
    
    import requests
    from io import StringIO
    # https://docs.google.com/spreadsheets/d/1LbILFyaTHeRPit8v3gwx2Db4uS1Hnx6dibeGHK9zXcU/edit?usp=sharing
    # LINK = 'https://docs.google.com/spreadsheet/ccc?key=1LbILFyaTHeRPit8v3gwx2Db4uS1Hnx6dibeGHK9zXcU&output=csv'
    if link is None:
        link = 'https://docs.google.com/spreadsheet/ccc?key=1XvVDFZSvcWWyVaLaQuTpglOqrCGB6Kdf6c78JJxymYw&output=csv'
    response = requests.get(link)
    assert response.status_code == 200, 'Wrong status code'
    loci_mapping_df = pd.read_csv(
        StringIO(response.content.decode()),
        sep=","
    ).set_index("region")
    
    return loci_mapping_df

In [ ]:
loci_mapping_old = fetch_loci_mapping('https://docs.google.com/spreadsheet/ccc?key=1LbILFyaTHeRPit8v3gwx2Db4uS1Hnx6dibeGHK9zXcU&output=csv')
loci_mapping_old = loci_mapping_old[~ (loci_mapping_old["duplicated"] == "YES")]

loci_mapping_new = fetch_loci_mapping()
loci_mapping_new = loci_mapping_new[~ (loci_mapping_new["duplicated"] == "YES")]

In [ ]:
loci_df = loci_mapping_new.iloc[:, :1]

In [ ]:
loci_df.index = loci_mapping_new.index

In [ ]:
lved = (loci_mapping_old.suggestive_significance == "YES").apply(lambda x: "SUG" if x else "YES")

In [ ]:
loci_df = loci_df.merge(lved, left_index=True, right_index=True, how="outer")
loci_df = loci_df.rename({"suggestive_significance": "LVED"}, axis=1)

In [ ]:
new_loci_df = pd.read_csv("../00_CardiacMotion/analysis/loci_pvals_static_vs_dynamic.csv")
static_loci = new_loci_df.loc[new_loci_df.variable_type == "static", ["region", "min_P"]]
dynamic_loci = new_loci_df.loc[new_loci_df.variable_type == "dynamic", ["region", "min_P"]]

In [ ]:
loci_df = loci_df.merge(static_loci, left_index=True, right_on="region", how="left")
# loci_df = loci_df.merge(static_loci, left_index=True, right_on="region", how="left")
loci_df = loci_df.rename({"min_P": "static"}, axis=1)
loci_df = loci_df.reset_index(drop=True)
loci_df = loci_df.merge(dynamic_loci, on="region", how="left").rename({"min_P": "dynamic"}, axis=1)
loci_df = loci_df[["region", "candidate_gene", "LVED", "static", "dynamic"]]

Loci that are found for dynamic variables

In [ ]:
sum(~loci_df.dynamic.isnull() & (loci_df.static.isnull() & loci_df.LVED.isnull()))

Loci that were study-wide significant for LVED (previous) but are not found for static variables

In [ ]:
sum((loci_df.LVED == "YES") & (loci_df.static.isnull()))

Loci that were suggestive for LVED and are not found for static variables

In [ ]:
sum((loci_df.LVED == "SUG") & (loci_df.static.isnull()))

___

In [ ]:
static_loci.head(20)

In [ ]:
dynamic_loci.head(20)